# 2D Toric Code — ED + NQS, with a symmetric-vs-non-symmetric CNN comparison

Perturbed 2D toric / surface code `H = -J Σ_v A_v - J Σ_p B_p - h_x Σ σˣ - h_z Σ σᶻ`,
reusing the repo's modules (`model/`, `simulation/`).

**Three architectures** (Step 3 trains all three and overlays them):

| name | KernelManager (geometry-exact conv)? | A_v / star symmetry? | what it is |
|---|---|---|---|
| `Combo` | ✅ | ✅ (Wilson 4-product) | the repo's `main.py` ansatz: edge-CNN → Wilson → invariant CNN → mean |
| `CNN_km_nosym` | ✅ | ❌ | same KernelManager edge-CNN, **Wilson removed**, dense readout |
| `VanillaCNN` | ❌ (plain `nn.Conv`) | ❌ | edges folded to `(Lx,Ly,2)` grid + standard conv (no KernelManager) |

So `Combo` vs `CNN_km_nosym` isolates the **symmetry**; `CNN_km_nosym` vs
`VanillaCNN` isolates the **geometry-exact kernel** — the 2D analog of the
`Three_TC` ablation (`ToricCNN_full` / `VanillaWilsonCNN` / `VanillaCNN`).

> The **2D repo has no built-in non-symmetric CNN** (only `Combo`/`RPP`, both
> Wilson-based); the two baselines here are defined in the notebook. The **3D**
> package (`Three_TC/model/networks.py`) does ship `VanillaCNN` / `VanillaWilsonCNN`.

**How to use**
1. Cell 1 (clone) → cell 2 (install — **auto-restarts the runtime once**, expected;
   after restart run from cell 3) → cell 3 (imports) → cell 4 (model zoo defs).
2. **Step 1 — ED**: set physics, get `E_exact`.
3. **Step 2 — single run** (live ΔE) or **Step 3 — compare all three** (combined plot).


In [ ]:
# 1. Clone (or update) the repo into the Colab VM. Idempotent: safe to re-run.
import os
REPO_URL = "https://github.com/SanzharBissenali/ThreeD_TC.git"
REPO_DIR = "/content/ThreeD_TC"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git pull --ff-only || true
print("cwd:", os.getcwd())

In [ ]:
# 2. Install the NQS stack.
#    Do NOT pin numpy/scipy/numba (Colab ships a consistent build; pinning numpy
#    triggers "cannot import name '_center' from numpy._core.umath").
import os
SENTINEL = "/content/.nqs_deps_done"          # on-disk flag (survives the restart)
if not os.path.exists(SENTINEL):
    !pip install -q jax==0.5.2 jaxlib==0.5.1 netket==3.16.1.post1 flax==0.10.4
    open(SENTINEL, "w").close()
    print(">>> Deps installed. Restarting runtime... after it restarts, run from cell 3.")
    os.kill(os.getpid(), 9)                    # auto-restart so new pkgs load cleanly
else:
    print("Deps already installed - continue from cell 3.")

In [ ]:
# 3. Imports + float64 (NetKet needs x64; set before importing jax/netket).
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import netket as nk
import flax.linen as fnn
from functools import partial
from scipy.sparse.linalg import eigsh
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from model.geometry import ToricCodeGeometry
from model.hamiltonian import create_hamiltonian
from model.exact_diag import hamiltonian_linop
from model.networks import KernelManager, create_model, CNN_noninvariant, Sequential
from simulation.custom_sampler import create_custom_sampler

print("jax", jax.__version__, "| netket", nk.__version__, "| backend:", jax.default_backend())

## Model zoo + training helpers (run once)

Defines the two non-symmetric baselines and `build_model` / `build_vs` /
`sr_train`, all reusing the repo's blocks. `sr_train` is the repo's SR step
(dense QGT + `pinv_smooth`) and records `E, ΔE, R̂, τ_corr, V-score, mcmc_acc`.

In [ ]:
# 4. Architectures + helpers.

# --- non-symmetric readout: dense head on edge features (breaks A_v invariance) ---
class DenseHead(fnn.Module):
    hidden: int
    @fnn.compact
    def __call__(self, x):                      # x: (channels, N)
        x = x.reshape(-1)
        x = fnn.tanh(fnn.Dense(self.hidden, param_dtype=jnp.float64)(x))
        return jnp.sum(fnn.Dense(1, param_dtype=jnp.float64)(x))   # real scalar log psi

# --- plain grid CNN (no KernelManager): fold N edges -> (Lx,Ly,2) grid ----------
def edge_grid_index(geo):
    """Missing OBC cells -> dummy slot N (filled with spin 0). PBC wraps cleanly."""
    Lx, Ly = geo.Lx, geo.Ly
    idx = np.full((Lx, Ly, 2), geo.N, dtype=np.int64)
    for q, (cx, cy) in enumerate(geo.arr_coord):
        if int(round(2 * cx)) % 2 == 1:                 # half-integer x -> horizontal edge
            col, row, ch = int(np.floor(cx)) % Lx, int(round(cy)) % Ly, 0
        else:                                           # half-integer y -> vertical edge
            col, row, ch = int(round(cx)) % Lx, int(np.floor(cy)) % Ly, 1
        idx[col, row, ch] = q
    return idx

class VanillaCNN(fnn.Module):
    grid_idx: tuple; shape: tuple; pad: str = "SAME"
    hidden: int = 6; depth: int = 2; kernel_size: int = 3
    @fnn.compact
    def __call__(self, x):                              # x: (..., N) spins +-1
        idx = jnp.asarray(self.grid_idx).reshape(self.shape)
        lead = x.shape[:-1]
        xa = jnp.concatenate([x, jnp.zeros(x.shape[:-1] + (1,), x.dtype)], axis=-1)
        g = xa[..., idx].reshape((-1,) + self.shape).astype(jnp.float64)   # (B,Lx,Ly,2)
        ks = (self.kernel_size, self.kernel_size)
        for _ in range(self.depth):
            g = fnn.elu(fnn.Conv(self.hidden, ks, padding=self.pad, param_dtype=jnp.float64)(g))
        g = fnn.Conv(1, ks, padding=self.pad, param_dtype=jnp.float64)(g)
        return jnp.sum(g, axis=(1, 2, 3)).reshape(lead)  # (B,) real log psi

def build_model(arch, geo, channels_noninv, channels_inv, kernel_size, kernel_size_inv,
                rescale, hidden, depth, vanilla_kernel):
    bc = geo.bc
    if arch in ("Combo", "CNN_km_nosym"):
        km = KernelManager(geo.Lx, geo.Ly, bc, kernel_size, kernel_size_inv,
                           geo.arr_coord, geo.dg_p, geo.N)
    if arch == "Combo":                              # symmetric (km edge-CNN -> Wilson -> inv)
        cfg = dict(architecture="Combo", channels_noninv=channels_noninv,
                   channels_inv=channels_inv, rescale=rescale, dtype="float64",
                   bc=bc, Lx=geo.Lx, Ly=geo.Ly, kernel_size=kernel_size,
                   kernel_size_inv=kernel_size_inv)
        return create_model(cfg, geo.plaq_all, km)
    if arch == "CNN_km_nosym":                       # km edge-CNN, NO Wilson + dense head
        blocks = [CNN_noninvariant(ci, co, km, "float64")
                  for ci, co in zip(channels_noninv, channels_noninv[1:])]
        return Sequential(tuple(blocks + [DenseHead(hidden)]))   # tuple: Flax needs hashable
    if arch == "VanillaCNN":                         # plain grid conv, no km, no symmetry
        gi = edge_grid_index(geo)
        return VanillaCNN(tuple(gi.reshape(-1).tolist()), (geo.Lx, geo.Ly, 2),
                          "CIRCULAR" if bc == "PBC" else "SAME", hidden, depth, vanilla_kernel)
    raise ValueError(f"unknown arch: {arch}")

def build_vs(model, geo, hi, use_custom, n_samples, n_chains, n_sweeps, n_discard, chunk_size):
    if use_custom:
        sa = create_custom_sampler(geo, hi, dict(n_chains=n_chains, n_sweeps=n_sweeps))
    else:
        sa = nk.sampler.MetropolisSampler(hi, rule=nk.sampler.rules.LocalRule(),
                                          n_chains=n_chains, n_sweeps=n_sweeps, dtype=jnp.int8)
    return nk.vqs.MCState(sa, model, n_samples=n_samples,
                          n_discard_per_chain=n_discard, chunk_size=chunk_size)

def sr_train(vs, H, E_exact, n_iter, dt, diag_shift, label=""):
    """Repo SR step; returns per-iteration history. Prints live E / ΔE / R̂ / acc."""
    hist = {k: [] for k in ["iter", "E", "E_err", "deltaE", "R_hat", "tau_corr", "Vscore", "mcmc_acc"]}
    loop = tqdm(range(n_iter), desc=label)
    for step in loop:
        E, f = vs.expect_and_grad(H)
        S = vs.quantum_geometric_tensor(
            nk.optimizer.qgt.QGTJacobianDense(diag_shift=diag_shift, diag_scale=0.0))
        dtheta, _ = S.solve(partial(nk.optimizer.solver.pinv_smooth,
                                    rtol=1e-30, rtol_smooth=1e-30),
                            jax.tree.map(lambda x: -1.0 * x, f))
        vs.parameters = jax.tree.map(lambda a, b: a + dt * b, vs.parameters, dtheta)
        Em  = float(np.real(E.mean))
        dE  = abs(Em - E_exact) / abs(E_exact)
        acc = float(vs.sampler_state.n_accepted) / float(vs.sampler_state.n_steps)
        vsc = vs.hilbert.size * float(E.variance) / Em**2
        for key, val in zip(hist, [step, Em, float(E.error_of_mean), dE,
                                   float(E.R_hat), float(E.tau_corr), vsc, acc]):
            hist[key].append(val)
        if np.isnan(Em):
            break
        loop.set_description(f"{label} E={Em:.5f} dE={dE:.2e} R_hat={float(E.R_hat):.3f} acc={acc:.3f}")
    return hist

print("model zoo ready: Combo | CNN_km_nosym | VanillaCNN")

## Step 1 — Exact diagonalization (sets `E_exact`, builds the NetKet Hamiltonian)

Edit the physics here. Small systems only (dim = 2ᴺ; keep N ≲ 22).

In [ ]:
# === EDIT: physical point (shared by every run below) =====================
L   = 3          # vertices per side
BC  = "OBC"      # "OBC" | "PBC"
hx  = 0.2        # sigma^x field
hy  = 0.0        # sigma^y field (must stay 0 here: real ansatz / stoquastic)
hz  = 0.2        # sigma^z field
J   = 1.0
# ==========================================================================
assert hy == 0.0, "These ansatze use a REAL log psi; keep hy = 0."

geo   = ToricCodeGeometry(L, L, BC)
N     = geo.N
DTYPE = "float64"
assert N <= 24, f"N={N} -> dim 2^{N} too large for ED in Colab."

H_ed, _ = hamiltonian_linop(geo, hx=hx, hy=hy, hz=hz, J=J)
ev = np.sort(eigsh(H_ed, k=2, which="SA", return_eigenvectors=False))
E_exact, gap = float(ev[0]), float(ev[1] - ev[0])

hi = nk.hilbert.Spin(s=1/2, N=N)
H  = create_hamiltonian(hi, geo.vertex_all, geo.plaq_all, geo.bonds,
                        hx=hx, hy=hy, hz=hz, J=J, dtype=DTYPE)

print(f"L={L} {BC} -> N={N} (dim 2^{N})   hx={hx} hz={hz}")
print(f"E_exact = {E_exact:.8f}   E/site = {E_exact/N:.6f}   gap = {gap:.6f}")

## Step 2 — single architecture, live ΔE

Pick one `ARCH` and watch it train. Set every hyperparameter here; Step 3 reuses
these same values.

In [ ]:
# === EDIT: architecture + hyperparameters =================================
ARCH            = "Combo"        # "Combo" | "CNN_km_nosym" | "VanillaCNN"

# -- shared / KernelManager archs (Combo, CNN_km_nosym) --
CHANNELS_NONINV = [1, 2, 4]      # non-invariant (edge) CNN channels
CHANNELS_INV    = [4, 4, 4]      # invariant CNN channels (Combo only)
KERNEL_SIZE     = 2              # non-inv kernel (taps; saturates at L-1)
KERNEL_SIZE_INV = L - 1          # invariant kernel (Combo only; k x k, max L-1)
RESCALE         = 10 ** 1.5
# -- non-symmetric heads --
HIDDEN          = 8              # DenseHead width (CNN_km_nosym) & VanillaCNN conv width
DEPTH           = 2              # VanillaCNN conv layers
VANILLA_KERNEL  = 3             # VanillaCNN conv kernel size

# -- optimization (SR) --
N_ITER          = 80
DT              = 7e-3
DIAG_SHIFT      = 1e-3
# -- Monte Carlo sampling (trimmed for speed: lower N_SAMPLES/N_ITER) --
N_SAMPLES       = 2048
N_CHAINS        = 16
N_SWEEPS        = N // 2
N_DISCARD       = 8
CHUNK_SIZE      = 2048
USE_CUSTOM_SAMPLER = True         # vertex-flip rule (single run only)
SEED            = 0
# ==========================================================================
assert "E_exact" in globals(), "Run Step 1 (ED) first."
np.random.seed(SEED)

model = build_model(ARCH, geo, CHANNELS_NONINV, CHANNELS_INV, KERNEL_SIZE,
                    KERNEL_SIZE_INV, RESCALE, HIDDEN, DEPTH, VANILLA_KERNEL)
vs = build_vs(model, geo, hi, USE_CUSTOM_SAMPLER, N_SAMPLES, N_CHAINS,
              N_SWEEPS, N_DISCARD, CHUNK_SIZE)
print(f"ARCH={ARCH}  n_params={vs.n_parameters}  N={N}  E_exact={E_exact:.6f}")

hist = sr_train(vs, H, E_exact, N_ITER, DT, DIAG_SHIFT, label=ARCH)

t = slice(-15, None)
print(f"\n=== FINAL ({ARCH}, median of last 15) ===")
print(f"E_NQS={np.median(hist['E'][t]):.6f}  E_exact={E_exact:.6f}  "
      f"deltaE={np.median(hist['deltaE'][t]):.3e}")
print(f"R_hat={np.median(hist['R_hat'][t]):.3f}  V-score={np.median(hist['Vscore'][t]):.2e}  "
      f"mcmc_acc={np.median(hist['mcmc_acc'][t]):.3f}")

In [ ]:
# Single-run convergence (energy + ΔE).
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist["iter"], hist["E"]); ax[0].axhline(E_exact, color="k", ls="--", label="E_exact")
ax[0].set_xlabel("SR step"); ax[0].set_ylabel("energy"); ax[0].legend(); ax[0].set_title(ARCH)
ax[1].semilogy(hist["iter"], hist["deltaE"])
ax[1].set_xlabel("SR step"); ax[1].set_ylabel(r"$\Delta E$"); ax[1].grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

## Step 3 — compare all three architectures (one combined figure)

Trains `Combo`, `CNN_km_nosym`, `VanillaCNN` at the same physical point with the
**same** SR hyperparameters and the **same LocalRule sampler** (so the ansatz is
the only difference), then overlays them on three panels: energy, ΔE, V-score.
Reuses the knobs set in Step 2.

In [ ]:
# Train the three architectures (same hyperparameters, same LocalRule sampler).
ARCHS   = ["Combo", "CNN_km_nosym", "VanillaCNN"]
results = {}
for arch in ARCHS:
    np.random.seed(SEED)
    m  = build_model(arch, geo, CHANNELS_NONINV, CHANNELS_INV, KERNEL_SIZE,
                     KERNEL_SIZE_INV, RESCALE, HIDDEN, DEPTH, VANILLA_KERNEL)
    vsi = build_vs(m, geo, hi, False, N_SAMPLES, N_CHAINS, N_SWEEPS, N_DISCARD, CHUNK_SIZE)
    print(f"{arch:14s} n_params={vsi.n_parameters}")
    results[arch] = sr_train(vsi, H, E_exact, N_ITER, DT, DIAG_SHIFT, label=arch)

print("\n=== final deltaE (median of last 15 steps) ===")
for arch in ARCHS:
    h = results[arch]; t = slice(-15, None)
    print(f"  {arch:14s} deltaE={np.median(h['deltaE'][t]):.3e}  "
          f"V-score={np.median(h['Vscore'][t]):.2e}  mcmc_acc(local)={np.median(h['mcmc_acc'][t]):.3f}")

In [ ]:
# Combined figure: the three architectures on energy / ΔE / V-score.
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for arch in ARCHS:
    h = results[arch]
    ax[0].plot(h["iter"], h["E"], label=arch)
    ax[1].semilogy(h["iter"], h["deltaE"], label=arch)
    ax[2].semilogy(h["iter"], np.abs(h["Vscore"]), label=arch)
ax[0].axhline(E_exact, color="k", ls="--", lw=1, label="E_exact")
ax[0].set_ylabel("energy"); ax[0].set_title("energy")
ax[1].set_ylabel(r"$\Delta E=|E-E_{exact}|/|E_{exact}|$"); ax[1].set_title("relative error (log)")
ax[2].set_ylabel(r"V-score $=N\,\mathrm{Var}/E^2$"); ax[2].set_title("V-score (log)")
for a in ax:
    a.set_xlabel("SR step"); a.legend(fontsize=8); a.grid(True, which="both", alpha=0.3)
plt.suptitle(f"L={L} {BC},  hx={hx}  hz={hz}   —   symmetric (Combo) vs non-symmetric CNNs")
plt.tight_layout(); plt.show()

## Step 4 — `Combo` hyperparameter sweep (train, save, compare)

Trains the **`Combo`** architecture for a list of hyperparameter settings, saves
every run's curve to disk (the Colab VM is ephemeral), and the next cell overlays
them. Each `SWEEP` entry overrides a subset of the Step-2 defaults; everything
else is inherited — so set the physics in Step 1 and the baseline knobs in Step 2,
then list only what you vary here.

In [ ]:
# === EDIT: Combo hyperparameter sweep =====================================
# Each entry: a label + any keys to OVERRIDE. Tunable keys:
#   dt, diag_shift, n_iter, n_samples, n_chains, n_sweeps, n_discard, chunk_size,
#   kernel_size, kernel_size_inv, channels_noninv, channels_inv, use_custom_sampler
SWEEP = [
    dict(label="dt7e-3_ds1e-3", dt=7e-3, diag_shift=1e-3),
    dict(label="dt2e-2_ds1e-3", dt=2e-2, diag_shift=1e-3),
    dict(label="dt7e-3_ds1e-2", dt=7e-3, diag_shift=1e-2),
    dict(label="dt2e-2_ds3e-3", dt=2e-2, diag_shift=3e-3),
]
SAVE_PATH = "combo_sweep_results.json"
# ==========================================================================
import json
assert "E_exact" in globals(), "Run Step 1 (ED) first."

def _base():   # Step-2 knobs are the sweep baseline
    return dict(dt=DT, diag_shift=DIAG_SHIFT, n_iter=N_ITER, n_samples=N_SAMPLES,
                n_chains=N_CHAINS, n_sweeps=N_SWEEPS, n_discard=N_DISCARD,
                chunk_size=CHUNK_SIZE, kernel_size=KERNEL_SIZE,
                kernel_size_inv=KERNEL_SIZE_INV, channels_noninv=CHANNELS_NONINV,
                channels_inv=CHANNELS_INV, use_custom_sampler=USE_CUSTOM_SAMPLER)

sweep_results = {}
for entry in SWEEP:
    c = _base(); c.update({k: v for k, v in entry.items() if k != "label"})
    label = entry["label"]
    np.random.seed(SEED)
    model = build_model("Combo", geo, c["channels_noninv"], c["channels_inv"],
                        c["kernel_size"], c["kernel_size_inv"], RESCALE,
                        HIDDEN, DEPTH, VANILLA_KERNEL)
    vsi = build_vs(model, geo, hi, c["use_custom_sampler"], c["n_samples"],
                   c["n_chains"], c["n_sweeps"], c["n_discard"], c["chunk_size"])
    print(f"[{label}] n_params={vsi.n_parameters}  dt={c['dt']}  diag_shift={c['diag_shift']}  "
          f"n_samples={c['n_samples']}  n_iter={c['n_iter']}")
    h = sr_train(vsi, H, E_exact, c["n_iter"], c["dt"], c["diag_shift"], label=label)
    h["config"] = c
    sweep_results[label] = h

# persist to disk (download with the optional cell, or reload later)
with open(SAVE_PATH, "w") as _f:
    json.dump({"L": L, "BC": BC, "hx": hx, "hz": hz, "E_exact": E_exact,
               "results": sweep_results}, _f)
print(f"\nsaved {len(sweep_results)} runs -> {SAVE_PATH}")
print("=== final deltaE (median of last 15 steps) ===")
for label, h in sweep_results.items():
    print(f"  {label:16s} deltaE={np.median(h['deltaE'][-15:]):.3e}  "
          f"V-score={np.median(h['Vscore'][-15:]):.2e}  mcmc_acc={np.median(h['mcmc_acc'][-15:]):.3f}")

In [ ]:
# Compare hyperparameters: ΔE convergence (left) + final ΔE per config (right).
# Reuses `sweep_results` from the cell above; or reload from disk:
#   sweep_results = json.load(open(SAVE_PATH))["results"]
labels   = list(sweep_results.keys())
final_dE = [np.median(sweep_results[l]["deltaE"][-15:]) for l in labels]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
for l in labels:
    h = sweep_results[l]
    ax[0].semilogy(h["iter"], h["deltaE"], label=l)
ax[0].set_xlabel("SR step"); ax[0].set_ylabel(r"$\Delta E=|E-E_{exact}|/|E_{exact}|$")
ax[0].set_title("ΔE convergence"); ax[0].legend(fontsize=8); ax[0].grid(True, which="both", alpha=0.3)

ax[1].bar(range(len(labels)), final_dE)
ax[1].set_yscale("log"); ax[1].set_xticks(range(len(labels)))
ax[1].set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
ax[1].set_ylabel(r"final $\Delta E$ (median last 15)"); ax[1].set_title("final ΔE by config")
ax[1].grid(True, axis="y", which="both", alpha=0.3)

plt.suptitle(f"Combo hyperparameter sweep — L={L} {BC},  hx={hx}  hz={hz}")
plt.tight_layout(); plt.show()

best = labels[int(np.argmin(final_dE))]
print(f"best config: {best}  (deltaE={min(final_dE):.3e})")

## Notes

- **Where symmetry should win most:** topological / small-`hz` points, where A_v is
  a near-exact symmetry of the ground state. Deep in the trivial phase A_v is badly
  broken, so the symmetric ansatz's advantage shrinks. Try `(hx,hz)=(0.0,0.0)`,
  `(0.2,0.2)` vs `(0.2,1.0)` and re-run Step 3.
- **Fairness knobs:** the baselines' capacity is set by `HIDDEN` / `DEPTH` /
  `VANILLA_KERNEL` and the `CHANNELS_*`. Match `n_params` (printed) across archs for
  a clean comparison.
- **Sampler:** Step 3 uses LocalRule for all three (the star-flip rule is "free" only
  for A_v-invariant ψ, so using it would advantage `Combo`). Step 2 lets you toggle
  `USE_CUSTOM_SAMPLER` for a single run.
- **Repo status:** the 2D repo has no non-symmetric CNN; both baselines live in this
  notebook. The 3D `Three_TC` ships `VanillaCNN` / `VanillaWilsonCNN`.
